In [1]:
import pyspark
print(pyspark.__version__)

3.4.0


In [2]:
import os
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

#stop any existing Spark session to avoid conflicts
try:
    spark.stop()
    print("Pre-existing background Spark session stopped successfully.")
except NameError:
    pass

#vars from env file
if os.path.exists("../.env"):
    with open("../.env") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                clean_line = line.replace("export ", "").strip()
                key, val = clean_line.split("=", 1)
                os.environ[key.strip()] = val.strip().strip('"').strip("'")

#spark new session with AWS credentials and S3 configurations
spark = SparkSession.builder \
    .appName("Conebusters-Feature-Consolidation-Pipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .getOrCreate()

print("Fresh Spark Session successfully activated with active AWS keys!")

:: loading settings :: url = jar:file:/home/ec2-user/.local/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f8fdb2f3-f74f-41a1-a625-e86ce8553362;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 380ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------

Fresh Spark Session successfully activated with active AWS keys!


In [3]:
#s3 already processed datasets
S3_PROCESSED_TLC = "s3a://de300-project7/processed/tlc/"
S3_PROCESSED_ATC = "s3a://de300-project7/processed/atc/"

#two primary datasets
tlc_df = spark.read.parquet(S3_PROCESSED_TLC)
atc_df = spark.read.parquet(S3_PROCESSED_ATC)

print("Primary taxi and traffic tables successfully loaded from S3.")

26/05/31 16:27:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Primary taxi and traffic tables successfully loaded from S3.


In [4]:
#structural record counts
print(f"Total Preprocessed TLC Taxi Rows: {tlc_df.count()}")
print(f"Total Preprocessed ATC Traffic Rows: {atc_df.count()}")

print("\n================== TRAFFIC DATA (ATC) ==================")
atc_df.printSchema()
print("Sample row look:")
atc_df.select("SegmentID", "Boro", "street").show(3, truncate=False)

print("\n=================== TAXI DATA (TLC) ===================")
tlc_df.printSchema()

Total Preprocessed TLC Taxi Rows: 7799653
Total Preprocessed ATC Traffic Rows: 293265

================== TRAFFIC DATA (ATC) ==================
root
 |-- SegmentID: integer (nullable = true)
 |-- Boro: string (nullable = true)
 |-- street: string (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- HH: integer (nullable = true)
 |-- historical_median_volume: integer (nullable = true)

Sample row look:
+---------+-------------+-----------+
|SegmentID|Boro         |street     |
+---------+-------------+-----------+
|44       |Staten Island|MOON AVENUE|
|44       |Staten Island|MOON AVENUE|
|44       |Staten Island|MOON AVENUE|
+---------+-------------+-----------+
only showing top 3 rows


=================== TAXI DATA (TLC) ===================
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: dou

In [5]:
#location-based columns to check for borough words
print("TLC Spatial Columns:")
print([col for col in tlc_df.columns if "id" in col.lower() or "boro" in col.lower() or "zone" in col.lower()])

TLC Spatial Columns:
['VendorID', 'RatecodeID', 'PULocationID', 'DOLocationID']


In [6]:
#Group by Borough, Day of Week, and Hour to create a unified Borough Traffic Profile
borough_traffic_profile = atc_df.groupBy("Boro", "day_of_week", "HH") \
    .agg(
        F.round(F.avg("historical_median_volume")).alias("boro_avg_median_volume"),
        F.max("historical_median_volume").alias("boro_max_volume"),
        F.round(F.stddev("historical_median_volume")).alias("boro_volume_variance")
    )

print("Borough-level baseline traffic profile successfully compiled.")
print(f"Total profile lookup rows: {borough_traffic_profile.count()}")
borough_traffic_profile.show(5)

Borough-level baseline traffic profile successfully compiled.


Total profile lookup rows: 840


+-------------+-----------+---+----------------------+---------------+--------------------+
|         Boro|day_of_week| HH|boro_avg_median_volume|boro_max_volume|boro_volume_variance|
+-------------+-----------+---+----------------------+---------------+--------------------+
|Staten Island|          4| 18|                 122.0|            723|               127.0|
|Staten Island|          1|  8|                  46.0|            281|                49.0|
|Staten Island|          1| 10|                  81.0|            430|                87.0|
|Staten Island|          7| 14|                 130.0|            647|               138.0|
|    Manhattan|          1| 15|                 175.0|           1366|               197.0|
+-------------+-----------+---+----------------------+---------------+--------------------+
only showing top 5 rows



In [7]:
print("--- ENGINEERED TRAFFIC LOOKUP PROFILE SCHEMA ---")
borough_traffic_profile.printSchema()

--- ENGINEERED TRAFFIC LOOKUP PROFILE SCHEMA ---
root
 |-- Boro: string (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- HH: integer (nullable = true)
 |-- boro_avg_median_volume: double (nullable = true)
 |-- boro_max_volume: integer (nullable = true)
 |-- boro_volume_variance: double (nullable = true)



In [8]:
#spatial crosswalk

In [9]:
# official NYC TLC lookup table directly from the public data repository
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
pandas_lookup = pd.read_csv(url)

#to PySpark DataFrame
zone_lookup_df = spark.createDataFrame(pandas_lookup) \
    .select(F.col("LocationID"), F.col("Borough").alias("Boro_Name"))

print("Official NYC Spatial Crosswalk successfully generated.")
zone_lookup_df.show(5)

Official NYC Spatial Crosswalk successfully generated.


+----------+-------------+
|LocationID|    Boro_Name|
+----------+-------------+
|         1|          EWR|
|         2|       Queens|
|         3|        Bronx|
|         4|    Manhattan|
|         5|Staten Island|
+----------+-------------+
only showing top 5 rows



In [10]:
#example
zone_lookup_df.filter(F.col("LocationID") == 66).show()

+----------+---------+
|LocationID|Boro_Name|
+----------+---------+
|        66| Brooklyn|
+----------+---------+



In [11]:
#temporal keys from real-time pickup timestamp
prepared_tlc = tlc_df \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))

#map PULocationID to its Pickup Borough text name
tlc_with_pu_boro = prepared_tlc \
    .join(zone_lookup_df, prepared_tlc.PULocationID == zone_lookup_df.LocationID, "left") \
    .withColumnRenamed("Boro_Name", "pickup_boro") \
    .drop("LocationID")

#map DOLocationID to its Dropoff Borough text name
tlc_spatial_ready = tlc_with_pu_boro \
    .join(zone_lookup_df, tlc_with_pu_boro.DOLocationID == zone_lookup_df.LocationID, "left") \
    .withColumnRenamed("Boro_Name", "dropoff_boro") \
    .drop("LocationID")

print("Taxi trips successfully enriched with temporal keys and text borough names.")
tlc_spatial_ready.select("tpep_pickup_datetime", "pickup_hour", "pickup_day_of_week", "pickup_boro", "dropoff_boro").show(50)

Taxi trips successfully enriched with temporal keys and text borough names.


+--------------------+-----------+------------------+-----------+------------+
|tpep_pickup_datetime|pickup_hour|pickup_day_of_week|pickup_boro|dropoff_boro|
+--------------------+-----------+------------------+-----------+------------+
| 2026-03-01 00:37:43|          0|                 1|  Manhattan|   Manhattan|
| 2026-03-01 00:31:01|          0|                 1|  Manhattan|   Manhattan|
| 2026-03-01 00:02:27|          0|                 1|  Manhattan|   Manhattan|
| 2026-03-01 00:26:40|          0|                 1|     Queens|      Queens|
| 2026-03-01 00:54:45|          0|                 1|  Manhattan|   Manhattan|
| 2026-03-01 00:18:31|          0|                 1|  Manhattan|   Manhattan|
| 2026-03-01 00:50:41|          0|                 1|  Manhattan|   Manhattan|
| 2026-03-01 00:15:43|          0|                 1|   Brooklyn|      Queens|
| 2026-03-01 00:30:09|          0|                 1|  Manhattan|   Manhattan|
| 2026-03-01 00:25:34|          0|                 1

In [12]:
print("--- SPATIAL MAPPING AUDIT (CROSS-BOROUGH TRIPS) ---")
tlc_spatial_ready.filter(F.col("pickup_boro") != F.col("dropoff_boro")) \
    .select(
        F.col("PULocationID"), 
        F.col("pickup_boro"), 
        F.col("DOLocationID"), 
        F.col("dropoff_boro")
    ) \
    .show(10, truncate=False)

#sanity Check, ensuring no rows failed the join (which would result in nulls)
null_pickups = tlc_spatial_ready.filter(F.col("pickup_boro").isNull()).count()
null_dropoffs = tlc_spatial_ready.filter(F.col("dropoff_boro").isNull()).count()

--- SPATIAL MAPPING AUDIT (CROSS-BOROUGH TRIPS) ---


+------------+-----------+------------+------------+
|PULocationID|pickup_boro|DOLocationID|dropoff_boro|
+------------+-----------+------------+------------+
|26          |Brooklyn   |205         |Queens      |
|26          |Brooklyn   |100         |Manhattan   |
|26          |Brooklyn   |208         |Bronx       |
|29          |Brooklyn   |236         |Manhattan   |
|26          |Brooklyn   |28          |Queens      |
|26          |Brooklyn   |28          |Queens      |
|26          |Brooklyn   |107         |Manhattan   |
|29          |Brooklyn   |170         |Manhattan   |
|26          |Brooklyn   |205         |Queens      |
|29          |Brooklyn   |261         |Manhattan   |
+------------+-----------+------------+------------+
only showing top 10 rows



In [13]:
#join 1 -> match traffic profile against PICKUP location data
consolidated_step1 = tlc_spatial_ready.join(
    borough_traffic_profile.select(
        F.col("Boro").alias("pu_join_boro"),
        F.col("day_of_week").alias("pu_join_dow"),
        F.col("HH").alias("pu_join_hh"),
        F.col("boro_avg_median_volume").alias("pickup_avg_median_volume"),
        F.col("boro_max_volume").alias("pickup_max_volume"),
        F.col("boro_volume_variance").alias("pickup_volume_variance")
    ),
    (tlc_spatial_ready.pickup_boro == F.col("pu_join_boro")) &
    (tlc_spatial_ready.pickup_day_of_week == F.col("pu_join_dow")) &
    (tlc_spatial_ready.pickup_hour == F.col("pu_join_hh")),
    "left"
).drop("pu_join_boro", "pu_join_dow", "pu_join_hh")

#join 2 -> match traffic profile against DROPOFF location data
final_consolidated_df = consolidated_step1.join(
    borough_traffic_profile.select(
        F.col("Boro").alias("do_join_boro"),
        F.col("day_of_week").alias("do_join_dow"),
        F.col("HH").alias("do_join_hh"),
        F.col("boro_avg_median_volume").alias("dropoff_avg_median_volume"),
        F.col("boro_max_volume").alias("dropoff_max_volume")
    ),
    (consolidated_step1.dropoff_boro == F.col("do_join_boro")) &
    (consolidated_step1.pickup_day_of_week == F.col("do_join_dow")) &
    (consolidated_step1.pickup_hour == F.col("do_join_hh")),
    "left"
).drop("do_join_boro", "do_join_dow", "do_join_hh")

print("Data consolidation complete! Traffic metrics appended side-by-side.")

Data consolidation complete! Traffic metrics appended side-by-side.


In [14]:
print(f"Original Taxi Row Count: {tlc_df.count()}")
print(f"Consolidated Table Row Count: {final_consolidated_df.count()}")

Original Taxi Row Count: 7799653


Consolidated Table Row Count: 7799653


In [15]:
#verify double-join results
final_consolidated_df.select(
    "tpep_pickup_datetime",
    "pickup_boro",
    "pickup_avg_median_volume",
    "pickup_max_volume",
    "dropoff_boro",
    "dropoff_avg_median_volume",
    "dropoff_max_volume"
).show(10, truncate=False)

+--------------------+-----------+------------------------+-----------------+------------+-------------------------+------------------+
|tpep_pickup_datetime|pickup_boro|pickup_avg_median_volume|pickup_max_volume|dropoff_boro|dropoff_avg_median_volume|dropoff_max_volume|
+--------------------+-----------+------------------------+-----------------+------------+-------------------------+------------------+
|2026-03-01 00:30:09 |Manhattan  |125.0                   |1582             |Manhattan   |125.0                    |1582              |
|2026-03-01 00:05:51 |Manhattan  |125.0                   |1582             |Manhattan   |125.0                    |1582              |
|2026-03-01 00:02:26 |Manhattan  |125.0                   |1582             |Manhattan   |125.0                    |1582              |
|2026-03-01 00:53:06 |Manhattan  |125.0                   |1582             |Manhattan   |125.0                    |1582              |
|2026-03-01 00:20:47 |Manhattan  |125.0         

In [16]:
final_consolidated_df.filter(F.col("pickup_boro") != F.col("dropoff_boro")) \
    .select(
        "tpep_pickup_datetime",
        "pickup_boro", 
        "pickup_avg_median_volume",
        "dropoff_boro", 
        "dropoff_avg_median_volume"
    ).show(10, truncate=False)

+--------------------+-----------+------------------------+------------+-------------------------+
|tpep_pickup_datetime|pickup_boro|pickup_avg_median_volume|dropoff_boro|dropoff_avg_median_volume|
+--------------------+-----------+------------------------+------------+-------------------------+
|2026-03-01 07:23:39 |Brooklyn   |52.0                    |Queens      |51.0                     |
|2026-03-01 09:46:15 |Brooklyn   |81.0                    |Manhattan   |109.0                    |
|2026-03-01 10:42:17 |Brooklyn   |98.0                    |Bronx       |160.0                    |
|2026-03-01 10:11:02 |Brooklyn   |98.0                    |Manhattan   |132.0                    |
|2026-03-01 17:27:59 |Brooklyn   |122.0                   |Queens      |131.0                    |
|2026-03-01 17:32:16 |Brooklyn   |122.0                   |Queens      |131.0                    |
|2026-03-02 04:27:40 |Brooklyn   |25.0                    |Manhattan   |39.0                     |
|2026-03-0

In [17]:
#Feature Engineering

In [20]:
print("Columnsfor feature engineering:")
print(enriched_time_df.columns)

Columnsfor feature engineering:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'pickup_hour', 'pickup_day_of_week', 'pickup_boro', 'dropoff_boro', 'pickup_avg_median_volume', 'pickup_max_volume', 'pickup_volume_variance', 'dropoff_avg_median_volume', 'dropoff_max_volume', 'Is_Weekend', 'Hour_Bin']


In [19]:
#flag weekends ( dayofweek: 1 = Sunday, 7 = Saturday)
#categorize raw hours into operational business blocks
enriched_time_df = final_consolidated_df \
    .withColumn("Is_Weekend", F.when(F.col("pickup_day_of_week").isin(1, 7), 1).otherwise(0)) \
    .withColumn("Hour_Bin", 
        F.when((F.col("pickup_hour") >= 7) & (F.col("pickup_hour") < 10), "Morning_Rush")
         .when((F.col("pickup_hour") >= 10) & (F.col("pickup_hour") < 16), "Off_Peak_Day")
         .when((F.col("pickup_hour") >= 16) & (F.col("pickup_hour") < 20), "Evening_Rush")
         .otherwise("Late_Night_Off_Peak")
    )

print("Temporal feature categories successfully engineered.")
enriched_time_df.select("tpep_pickup_datetime", "Is_Weekend", "Hour_Bin").show(5)

Temporal feature categories successfully engineered.


+--------------------+----------+-------------------+
|tpep_pickup_datetime|Is_Weekend|           Hour_Bin|
+--------------------+----------+-------------------+
| 2026-03-01 00:02:26|         1|Late_Night_Off_Peak|
| 2026-03-01 00:53:06|         1|Late_Night_Off_Peak|
| 2026-03-01 00:20:47|         1|Late_Night_Off_Peak|
| 2026-03-01 00:16:11|         1|Late_Night_Off_Peak|
| 2026-03-01 00:19:33|         1|Late_Night_Off_Peak|
+--------------------+----------+-------------------+
only showing top 5 rows



In [21]:
#trip duration in minutes
#average velocity (miles per minute), guarding against division-by-zero errors
clean_base_df = enriched_time_df.localCheckpoint()

df_with_duration = clean_base_df.withColumn(
    "trip_duration", 
    (F.unix_timestamp(F.col("tpep_dropoff_datetime")) - F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 60.0
)

features_with_dynamics_df = df_with_duration.withColumn(
    "average_velocity",
    F.when(F.col("trip_duration") > 0, F.col("trip_distance") / F.col("trip_duration")).otherwise(0.0)
)

print("Trip dynamic tracking vectors successfully calculated!")
features_with_dynamics_df.select("trip_distance", "trip_duration", "average_velocity").show(5)

26/05/31 16:36:59 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/31 16:37:17 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 158.6 MiB so far)
26/05/31 16:37:17 WARN BlockManager: Persisting block rdd_198_0 to disk instead.
26/05/31 16:37:17 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 238.2 MiB so far)
26/05/31 16:37:17 WARN BlockManager: Persisting block rdd_198_1 to disk instead.
26/05/31 16:37:23 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 357.3 MiB so far)
26/05/31 16:37:26 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 66.6 MiB so far)
26/05/31 16:37:26 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 357.4 MiB so far)
26/05/31 16:37:26 WARN BlockManager: Persisting block rdd_198_2 to disk instead.
26/05/31 16:37:32 WARN 

Trip dynamic tracking vectors successfully calculated!
+-------------+------------------+-------------------+
|trip_distance|     trip_duration|   average_velocity|
+-------------+------------------+-------------------+
|         19.4|              60.0| 0.3233333333333333|
|          9.4| 81.16666666666667|0.11581108829568788|
|          1.8|15.583333333333334|0.11550802139037433|
|         13.5|             117.6|0.11479591836734694|
|         20.0|              80.9| 0.2472187886279357|
+-------------+------------------+-------------------+
only showing top 5 rows



26/05/31 16:37:40 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 103.2 MiB so far)


In [22]:
#cross-product of physical distance and expected pickup traffic congestion
final_feature_matrix = features_with_dynamics_df \
    .withColumn("traffic_distance_interaction", 
        F.col("trip_distance") * F.col("pickup_avg_median_volume")
    )

print("Engineered feature layer calculations complete.")
final_feature_matrix.select("Is_Weekend", "Hour_Bin", "trip_duration", "average_velocity", "traffic_distance_interaction").show(5)

Engineered feature layer calculations complete.
+----------+------------+------------------+-------------------+----------------------------+
|Is_Weekend|    Hour_Bin|     trip_duration|   average_velocity|traffic_distance_interaction|
+----------+------------+------------------+-------------------+----------------------------+
|         1|Morning_Rush|              60.0| 0.3233333333333333|                      1008.8|
|         1|Morning_Rush| 81.16666666666667|0.11581108829568788|                       761.4|
|         1|Morning_Rush|15.583333333333334|0.11550802139037433|                       145.8|
|         1|Off_Peak_Day|             117.6|0.11479591836734694|                      1323.0|
|         1|Off_Peak_Day|              80.9| 0.2472187886279357|                      1960.0|
+----------+------------+------------------+-------------------+----------------------------+
only showing top 5 rows



26/05/31 16:39:01 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 103.2 MiB so far)


In [23]:
print("=================== PRE-FILTER DISTRIBUTION INTEGRITY CHECK ===================")
final_feature_matrix.select(
    "trip_duration", 
    "average_velocity", 
    "traffic_distance_interaction"
).summary("mean", "min", "max").show()

=================== PRE-FILTER DISTRIBUTION INTEGRITY CHECK ===================


26/05/31 16:44:46 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 44.0 MiB so far)
26/05/31 16:44:46 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 44.0 MiB so far)
26/05/31 16:44:48 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:44:50 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05/31 16:44:51 WARN MemoryStore: Not enough space to cache rdd_198_5 in memory! (computed 103.3 MiB so far)


+-------+------------------+-------------------+----------------------------+
|summary|     trip_duration|   average_velocity|traffic_distance_interaction|
+-------+------------------+-------------------+----------------------------+
|   mean|17.384264611943035|0.17815444933986732|           515.7760679584071|
|    min|             -11.7|                0.0|                        0.07|
|    max| 7482.066666666667| 1124.3999999999999|                     20520.3|
+-------+------------------+-------------------+----------------------------+



In [24]:
#realistic operational guardrails to protect model training
production_ready_df = final_feature_matrix.filter(
    (F.col("trip_duration") >= 1.0) & (F.col("trip_duration") <= 180.0) &
    (F.col("average_velocity") >= 0.033) & (F.col("average_velocity") <= 1.33) &
    (F.col("trip_distance") > 0.0)
)

print("--- POST-FILTER INTEGRITY PROFILE ---")
production_ready_df.select("trip_duration", "average_velocity", "traffic_distance_interaction").summary("min", "max").show()

--- POST-FILTER INTEGRITY PROFILE ---


26/05/31 16:46:00 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 29.4 MiB so far)
26/05/31 16:46:00 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 66.6 MiB so far)
26/05/31 16:46:03 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:46:05 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05/31 16:46:06 WARN MemoryStore: Not enough space to cache rdd_198_5 in memory! (computed 103.3 MiB so far)


+-------+------------------+------------------+----------------------------+
|summary|     trip_duration|  average_velocity|traffic_distance_interaction|
+-------+------------------+------------------+----------------------------+
|    min|               1.0|             0.033|                        1.35|
|    max|179.96666666666667|1.3281343731253747|                    19727.51|
+-------+------------------+------------------+----------------------------+



In [25]:
#example trip
final_feature_matrix.select(
    "PULocationID",
    "pickup_boro",
    "pickup_avg_median_volume",
    "DOLocationID",
    "dropoff_boro",
    "dropoff_avg_median_volume",
    "traffic_distance_interaction"
).show(5, truncate=False)

+------------+-----------+------------------------+------------+------------+-------------------------+----------------------------+
|PULocationID|pickup_boro|pickup_avg_median_volume|DOLocationID|dropoff_boro|dropoff_avg_median_volume|traffic_distance_interaction|
+------------+-----------+------------------------+------------+------------+-------------------------+----------------------------+
|26          |Brooklyn   |52.0                    |205         |Queens      |51.0                     |1008.8                      |
|26          |Brooklyn   |81.0                    |100         |Manhattan   |109.0                    |761.4                       |
|26          |Brooklyn   |81.0                    |21          |Brooklyn    |81.0                     |145.8                       |
|26          |Brooklyn   |98.0                    |208         |Bronx       |160.0                    |1323.0                      |
|29          |Brooklyn   |98.0                    |236         |Manha

26/05/31 16:47:47 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 103.2 MiB so far)


In [26]:
print("=================== PRE-FLIGHT NULL VALUE COUNT SCAN ===================")

#dynamic count array to scan every critical feature channel
null_counts = production_ready_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "pickup_boro", 
        "dropoff_boro", 
        "pickup_avg_median_volume", 
        "dropoff_avg_median_volume", 
        "trip_duration", 
        "average_velocity", 
        "traffic_distance_interaction"
    ]
])

null_counts.show()

=================== PRE-FLIGHT NULL VALUE COUNT SCAN ===================


26/05/31 16:49:09 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 44.0 MiB so far)
26/05/31 16:49:09 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 44.0 MiB so far)
26/05/31 16:49:11 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:49:13 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05/31 16:49:14 WARN MemoryStore: Not enough space to cache rdd_198_5 in memory! (computed 103.3 MiB so far)


+-----------+------------+------------------------+-------------------------+-------------+----------------+----------------------------+
|pickup_boro|dropoff_boro|pickup_avg_median_volume|dropoff_avg_median_volume|trip_duration|average_velocity|traffic_distance_interaction|
+-----------+------------+------------------------+-------------------------+-------------+----------------+----------------------------+
|          0|           0|                   12172|                    62531|            0|               0|                       12172|
+-----------+------------+------------------------+-------------------------+-------------+----------------+----------------------------+



In [27]:
print("=================== GLOBAL PRE-FLIGHT ALL-COLUMN NULL SCAN ===================")

#generate a null count for every single column in the dataframe
global_null_counts = production_ready_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in production_ready_df.columns
])

#vertical wrapping
global_null_counts.show(vertical=True)

=================== GLOBAL PRE-FLIGHT ALL-COLUMN NULL SCAN ===================


26/05/31 16:54:44 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 44.0 MiB so far)
26/05/31 16:54:44 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 44.0 MiB so far)
26/05/31 16:54:48 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:54:52 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05/31 16:54:53 WARN MemoryStore: Not enough space to cache rdd_198_5 in memory! (computed 103.3 MiB so far)


-RECORD 0-----------------------------
 VendorID                     | 0     
 tpep_pickup_datetime         | 0     
 tpep_dropoff_datetime        | 0     
 passenger_count              | 0     
 trip_distance                | 0     
 RatecodeID                   | 0     
 store_and_fwd_flag           | 0     
 PULocationID                 | 0     
 DOLocationID                 | 0     
 payment_type                 | 0     
 fare_amount                  | 0     
 extra                        | 0     
 mta_tax                      | 0     
 tip_amount                   | 0     
 tolls_amount                 | 0     
 improvement_surcharge        | 0     
 total_amount                 | 0     
 congestion_surcharge         | 0     
 Airport_fee                  | 0     
 cbd_congestion_fee           | 0     
 pickup_hour                  | 0     
 pickup_day_of_week           | 0     
 pickup_boro                  | 0     
 dropoff_boro                 | 0     
 pickup_avg_median_volume

In [28]:
#traffic columns that captured out-of-bounds nulls
critical_traffic_columns = [
    "pickup_avg_median_volume", 
    "dropoff_avg_median_volume"
]

#drop rows only if they failed the space-time traffic mapping profiles
final_clean_df = production_ready_df.dropna(subset=critical_traffic_columns)

print("=================== FINAL MASTER CHECKPOINT ===================")
print(f"Rows safely filtered out (out-of-bounds mapping): {production_ready_df.count() - final_clean_df.count()}")
print(f"Total pristine, 100% dense rows ready for S3: {final_clean_df.count()}")

=================== FINAL MASTER CHECKPOINT ===================


26/05/31 16:56:49 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 44.0 MiB so far)
26/05/31 16:56:49 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 44.0 MiB so far)
26/05/31 16:56:51 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:56:52 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05/31 16:56:54 WARN MemoryStore: Not enough space to cache rdd_198_5 in memory! (computed 103.3 MiB so far)
26/05/31 16:56:55 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 44.0 MiB so far)
26/05/31 16:56:55 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 44.0 MiB so far)
26/05/31 16:56:57 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:56:59 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05

Rows safely filtered out (out-of-bounds mapping): 69767


26/05/31 16:57:03 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 29.4 MiB so far)
26/05/31 16:57:03 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 103.2 MiB so far)
26/05/31 16:57:05 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:57:06 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05/31 16:57:07 WARN MemoryStore: Not enough space to cache rdd_198_5 in memory! (computed 103.3 MiB so far)


Total pristine, 100% dense rows ready for S3: 7531109


In [29]:
#S3 path for the master feature repo
S3_CONSOLIDATED_OUTPUT = "s3a://de300-project7/processed/consolidated_features/"

print(f"Initiating cloud commit. Writing compressed Parquet partitions to: {S3_CONSOLIDATED_OUTPUT}")

#pristine dataframe out to the shared storage layer
final_clean_df.write \
    .mode("overwrite") \
    .parquet(S3_CONSOLIDATED_OUTPUT)

print("\nSUCCESS! Data consolidation and feature engineering pipeline executed flawlessly.")
print("The master feature store is completely unblocked and live on AWS S3.")

Initiating cloud commit. Writing compressed Parquet partitions to: s3a://de300-project7/processed/consolidated_features/


26/05/31 16:59:12 WARN MemoryStore: Not enough space to cache rdd_198_0 in memory! (computed 44.0 MiB so far)
26/05/31 16:59:12 WARN MemoryStore: Not enough space to cache rdd_198_1 in memory! (computed 44.0 MiB so far)
26/05/31 16:59:31 WARN MemoryStore: Not enough space to cache rdd_198_2 in memory! (computed 103.3 MiB so far)
26/05/31 16:59:50 WARN MemoryStore: Not enough space to cache rdd_198_4 in memory! (computed 103.3 MiB so far)
26/05/31 16:59:51 WARN MemoryStore: Not enough space to cache rdd_198_5 in memory! (computed 3.6 MiB so far)



SUCCESS! Data consolidation and feature engineering pipeline executed flawlessly.
The master feature store is completely unblocked and live on AWS S3.
